In [ ]:
%%capture
!pip install boto3==1.35.55 botocore==1.35.55

# Dataset download 

In [1]:
%%capture
# PPE dataset download 
!rm dataset.zip
dataset_url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip"
!wget -O dataset.zip $dataset_url

@dataset{Dalvi_Construction_PPE_Dataset_2025,
    author    = {Mrunmayee Dalvi and Niyati Singh and Sahil Bhingarde and Ketaki Chalke},
    title     = {Construction-PPE: Personal Protective Equipment Detection Dataset},
    month     = {January},
    year      = {2025},
    version   = {1.0.0},
    license   = {AGPL-3.0},
    url       = {https://docs.ultralytics.com/datasets/detect/construction-ppe/},
    publisher = {Ultralytics}
}

In [7]:
import os
import boto3
import botocore

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = 'storage'

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more connection variables are empty.  "
                     "Please check your connection to an S3 bucket.")

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)


def upload_directory_to_s3(local_directory, s3_prefix):
    num_files = 0
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)
            num_files += 1
    return num_files
def upload_file_to_s3(local_file_path, s3_prefix):
    """Upload a single file to S3 bucket with specified prefix"""
    try:
        filename = os.path.basename(local_file_path)
        s3_key = os.path.join(s3_prefix, filename)
        bucket.upload_file(local_file_path, s3_key)
        print(f"{local_file_path} -> {s3_key}")
        return True
    except Exception as e:
        print(f"Error uploading file: {e}")
        return False
        
def download_file_from_s3(s3_key, local_file_path):
    """Download a single file from S3 bucket"""
    try:
        bucket.download_file(s3_key, local_file_path)
        print(f"{s3_key} -> {local_file_path}")
        return True
    except Exception as e:
        print(f"Error downloading file: {e}")
        return False


def delete_file_from_s3(s3_key):
    """Delete a single file from S3 bucket"""
    try:
        s3_object = bucket.Object(s3_key)
        s3_object.delete()
        print(f"Deleted: {s3_key}")
        return True
    except Exception as e:
        print(f"Error deleting file: {e}")
        return False
        
def list_objects(prefix):
    filter = bucket.objects.filter(Prefix=prefix)
    for obj in filter.all():
        print(obj.key)

In [9]:
list_objects("datasets")

datasets/dataset.zip


In [4]:
!rm -rf datasets

In [8]:
upload_file_to_s3('dataset.zip', 'datasets')
# This uploads file.txt to: my-prefix/file.txt

dataset.zip -> datasets/dataset.zip


True

In [10]:
list_objects("datasets")

datasets/dataset.zip
